Tạo spark session:

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ModelTraining") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

In [2]:
df_final = spark.read.parquet('/home/jovyan/data/features/final_dataset')
df_final.count()

2779603

In [3]:
from pyspark.sql.functions import isnan, when, count, col
df_final = df_final.filter(col('product_id').isNotNull())
df_final = df_final.fillna(-1, subset=['favorite_department', 'favorite_aisles'])
df_final = df_final.fillna(0)
df_final.select([count(when(col(c).isNull(), c)).alias(c) for c in df_final.columns]).show()

+-------+----------+-----+------------+-----------------------+---------------+--------------------+-------------+------------+-------------------+---------------+--------------------+--------------------+--------------------+---------------+---------------+---------------+-------------+---------------------+-------------+
|user_id|product_id|label|total_orders|avg_days_between_orders|avg_basket_size|overall_reorder_rate|favorite_hour|favorite_day|favorite_department|favorite_aisles|product_total_orders|product_reorder_rate|product_unique_users|up_times_bought|up_reorder_rate|up_avg_cart_pos|up_last_order|orders_since_last_buy|up_order_rate|
+-------+----------+-----+------------+-----------------------+---------------+--------------------+-------------+------------+-------------------+---------------+--------------------+--------------------+--------------------+---------------+---------------+---------------+-------------+---------------------+-------------+
|      0|         0|    0

In [6]:
df_final.columns

['user_id',
 'product_id',
 'label',
 'total_orders',
 'avg_days_between_orders',
 'avg_basket_size',
 'overall_reorder_rate',
 'favorite_hour',
 'favorite_day',
 'favorite_department',
 'favorite_aisles',
 'product_total_orders',
 'product_reorder_rate',
 'product_unique_users',
 'up_times_bought',
 'up_reorder_rate',
 'up_avg_cart_pos',
 'up_last_order',
 'orders_since_last_buy',
 'up_order_rate']

In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=['total_orders','avg_days_between_orders','avg_basket_size','overall_reorder_rate',\
 'favorite_hour','favorite_day','favorite_department','favorite_aisles', \
 'product_total_orders', 'product_reorder_rate','product_unique_users', \
 'up_times_bought','up_reorder_rate','up_avg_cart_pos','up_last_order','orders_since_last_buy', 'up_order_rate'],
    outputCol='features'
)
df_final = assembler.transform(df_final)

In [9]:
df_positive = df_final.filter(col('label') == 1)
df_negative = df_final.filter(col('label') == 0).sample(fraction=0.12, seed=42)

df_balanced = df_positive.union(df_negative)
df_balanced.groupBy('label').count().show()

+-----+------+
|label| count|
+-----+------+
|    1|272603|
|    0|276132|
+-----+------+



In [5]:
df_final.groupBy('label').count().show()


+-----+-------+
|label|  count|
+-----+-------+
|    1| 272603|
|    0|2300791|
+-----+-------+



In [10]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    labelCol='label',
    featuresCol='features',
    numTrees=25,
    seed=42
)

In [11]:
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(
    labelCol='label',
    featuresCol='features',
    maxIter=50,
    seed=42
)

In [12]:
train, test = df_balanced.randomSplit([0.8, 0.2], seed=42)
model = gbt.fit(train)


In [13]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
predictions = model.transform(test)
for metric in ['accuracy', 'precisionByLabel', 'recallByLabel', 'f1']:
    evaluator = MulticlassClassificationEvaluator(
        labelCol='label',
        predictionCol='prediction',
        metricName=metric
    )
    print(f'{metric}: {evaluator.evaluate(predictions)}')

accuracy: 0.6775836245631552
precisionByLabel: 0.6950923787528869
recallByLabel: 0.6473392719516785
f1: 0.6773378968310106


In [14]:
model.save('/home/jovyan/instacart-reorder-prediction/models/gbt_model')
print('Model saved!')

Model saved!


AUC: 0.7430630407509855
F1: 0.6773378968310106


In [24]:
predictions.groupBy('label', 'prediction').count().show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|       1.0|38174|
|    1|       0.0|16198|
|    0|       0.0|35506|
|    0|       1.0|20287|
+-----+----------+-----+

